# **RelayVSR for Video Upscaling**

- **Github page**: https://github.com/kopperx/RelayVSR
- **Notebook source**: https://github.com/Isi-dev/Google-Colab_Notebooks

In [13]:
# @title ⚙️ 1. Setup ComfyUI & Download 2x/4x Models
import os

from IPython.display import clear_output

!apt-get -q install -y aria2
# Clone fresh ComfyUI for native RelayVSR support
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# FIX: Install all official ComfyUI requirements to catch comfy-kitchen and comfy-aimdo
!pip install -r requirements.txt
# Install the extra video-specific libraries needed for RelayVSR decoding
!pip install -q decord imageio[ffmpeg] av diffusers accelerate

os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)

print("Downloading 2x LightX2V Model & VAE...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/diffusion_models/relayvsr_x2_lightx2v_int8_convrot.safetensors -d models/diffusion_models -o relayvsr_x2_lightx2v_int8_convrot.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/vae/relayvsr_flash_vae_x2_fp32.safetensors -d models/vae -o relayvsr_flash_vae_x2_fp32.safetensors

print("Downloading 4x Original Model & VAE...")
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/diffusion_models/relayvsr_x4_int8_convrot.safetensors -d models/diffusion_models -o relayvsr_x4_int8_convrot.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/RelayVSR/resolve/main/vae/relayvsr_flash_vae_x4_fp32.safetensors -d models/vae -o relayvsr_flash_vae_x4_fp32.safetensors

clear_output()
print("✅ Environment Setup Complete!")

✅ Environment Setup Complete!


In [11]:
# @title 📂 2. Upload Source Video
import os
from google.colab import files
from IPython.display import clear_output, HTML, display
from base64 import b64encode

# @markdown --- Upload Settings ---
display_input_video = True # @param {type:"boolean"}

input_dir = "/content/ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)
%cd {input_dir}

print("Upload your video...")
uploaded = files.upload()
clear_output()

# Capture the last uploaded file's name and path
uploaded_filename = list(uploaded.keys())[-1]
input_video_path = os.path.join(input_dir, uploaded_filename)

print(f"✅ Ready for processing: {uploaded_filename}")

# Optional In-Notebook Display for Input Video
if display_input_video:
    print("Loading input video preview...")
    video_data = open(input_video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()
    display(HTML(f"""
    <video width="100%" style="max-width: 600px;" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))

%cd /content/ComfyUI

✅ Ready for processing: wan21 surprised and shocked.mp4
Loading input video preview...


/content/ComfyUI


In [15]:
# @title 🚀 3. Upscale Video
import sys
import os
import torch
import decord
import numpy as np
import imageio
import inspect
import subprocess
from IPython.display import HTML, display
from base64 import b64encode

# @markdown --- Execution Settings ---
scale = "4x (Original)" # @param ["2x (LightX2V)", "4x (Original)"]
detail = 1.0 # @param {type:"slider", min:0.0, max:1.0, step:0.05}
seed = 42 # @param {type:"integer"}
display_output_video = True # @param {type:"boolean"}

# Inherit paths directly from Cell 2
input_video = input_video_path
output_video = f"/content/ComfyUI/output/upscaled_{uploaded_filename}"

if "2x" in scale:
    model_name = "relayvsr_x2_lightx2v_int8_convrot.safetensors"
    vae_name = "relayvsr_flash_vae_x2_fp32.safetensors"
else:
    model_name = "relayvsr_x4_int8_convrot.safetensors"
    vae_name = "relayvsr_flash_vae_x4_fp32.safetensors"

# 1. Initialize Native Environment
sys.path.insert(0, '/content/ComfyUI')
import nodes
import comfy.ldm.wan.model_vsr
import comfy.model_management

mappings = nodes.NODE_CLASS_MAPPINGS

print(f"Loading {scale} models...")
ModelLoader = mappings["UNETLoader"]()
VAELoader = mappings["VAELoader"]()

# Load weights using the standard loaders
model = getattr(ModelLoader, getattr(ModelLoader, 'FUNCTION', 'load_unet'))(unet_name=model_name, weight_dtype="default")[0]
vae = getattr(VAELoader, getattr(VAELoader, 'FUNCTION', 'load_vae'))(vae_name=vae_name)[0]

print(f"Decoding {input_video} into PyTorch tensor...")
vr = decord.VideoReader(input_video, ctx=decord.cpu(0))
fps = vr.get_avg_fps()
frames = vr.get_batch(range(len(vr))).asnumpy()
tensor_frames = torch.from_numpy(frames).float() / 255.0

with torch.inference_mode():
    print("Generating conditioning and empty latents ...")
    # 2. Direct Conditioning implementation (bypassing RelayVSRConditioning node)
    diffusion_model = model.model.diffusion_model
    keyframes = comfy.ldm.wan.model_vsr.relay_keyframes(tensor_frames.shape[0])

    positive = [[diffusion_model.positive_conditioning.unsqueeze(0).clone(), {"lq_images": tensor_frames[keyframes]}]]
    empty_negative = []

    latent_size = comfy.ldm.wan.model_vsr.relay_latent_size(tensor_frames.shape[1], tensor_frames.shape[2], diffusion_model.latent_stride)
    latent = torch.zeros([1, model.model.latent_format.latent_channels, len(keyframes), *latent_size], device=comfy.model_management.intermediate_device())
    latent_dict = {"samples": latent}

    print("Sampling keyframes through Wan backbone...")
    # 3. KSampler (Standard Node API still works here)
    SamplerClass = mappings["KSampler"]()
    sampler_func = getattr(SamplerClass, getattr(SamplerClass, 'FUNCTION', 'sample'))
    sampled_latent = sampler_func(
        model=model,
        seed=seed,
        steps=1,
        cfg=1.0,
        sampler_name="euler",
        scheduler="normal",
        positive=positive,
        negative=empty_negative,
        latent_image=latent_dict,
        denoise=1.0
    )[0]

    print(f"Decoding full sequence with detail={detail} ...")
    # 4. Direct Decode implementation (bypassing VSRDecode node)
    decoded = vae.decode(sampled_latent["samples"], vae_options={"lq_images": tensor_frames, "detail": detail})
    upscaled_tensor = decoded.reshape(-1, *decoded.shape[-3:])

# 5. Save & Remux Audio
temp_vid = "/content/ComfyUI/output/temp_out.mp4"
out_frames = (upscaled_tensor.cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)

writer = imageio.get_writer(temp_vid, fps=fps, codec='libx264', quality=8, pixelformat='yuv420p')
for frame in out_frames: writer.append_data(frame)
writer.close()

subprocess.run([
    "ffmpeg", "-y", "-i", temp_vid, "-i", input_video,
    "-c:v", "copy", "-c:a", "aac", "-map", "0:v:0", "-map", "1:a:0?",
    "-shortest", output_video
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.remove(temp_vid)

print(f"✅ Upscale complete: {output_video}")

# 6. In-Notebook Display
if display_output_video:
    print("Loading video preview...")
    video_data = open(output_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()
    display(HTML(f"""
    <video width="100%" style="max-width: 800px;" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))

Loading 4x (Original) models...
Decoding /content/ComfyUI/input/wan21 surprised and shocked.mp4 into PyTorch tensor...
Generating conditioning and empty latents ...
Sampling keyframes through Wan backbone...


  0%|          | 0/1 [00:00<?, ?it/s]

RelayVSR:   0%|          | 0/30 [00:00<?, ?block/s]

Decoding full sequence with detail=1.0 ...


RelayVSR Flash decode:   0%|          | 0/49 [00:00<?, ?frame/s]

✅ Upscale complete: /content/ComfyUI/output/upscaled_wan21 surprised and shocked.mp4
Loading video preview...
